In [ ]:
import numpy as np
import sys
sys.path.append("../../../")
sys.path.append("../../../Visualization/")
sys.path.append("../../../../")

In [ ]:
experiment_file = '../../../experiments/parallelized_experiments/output/double_zigzag_dash/2024_01_08_18_15/experiment_result.json'

stiffness_path = '../../../experiments/parallelized_experiments/output/double_zigzag_dash/2024_01_08_18_15'
name = 'double_zigzag_dash'

In [ ]:
import MeshFEM, visualization

In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [ ]:
m = MeshFEM.mesh.Mesh('../../../experiments/parallelized_experiments/output/double_zigzag_dash/2024_01_08_18_15/1.00_60.00/mesh_double_zigzag_dash_1.00_60.00.obj')
fusing_vtx = np.load('../../../experiments/parallelized_experiments/output/double_zigzag_dash/2024_01_08_18_15/1.00_60.00/fusedVtx_double_zigzag_dash_1.00_60.00.npy')

visualization.plot_2d_mesh(m, pointList=fusing_vtx, width=5, height=5)

### Overview

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np
import visualize_stiffness

In [ ]:
with open(experiment_file, 'r') as fp:
    data = json.load(fp)

In [ ]:
df = pd.DataFrame(data['data'])

In [ ]:
valid_tags = np.array(df['name'][df['Planar equilibrium'] == 1])

In [ ]:
invalid_tags = np.array(df['name'][df['Planar equilibrium'] != 1])

In [ ]:
invalid_tags

In [ ]:
df['Simulation Kappa value'][np.array(df['Planar equilibrium']) != 1]

In [ ]:
kappa_path = None

In [ ]:
radius = np.array(data['pattern_parameters'][0]['values'])
angles = np.array(data['pattern_parameters'][1]['values'])

In [ ]:
x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, name, valid_tags)

In [ ]:
new_valid_tags = []
for tag in valid_tags:
    if float(tag.split('_')[1]) != 45:
        new_valid_tags.append(tag)
        
valid_tags = np.array(new_valid_tags)
if angles[0] == 45:
    angles = angles[1:]

In [ ]:
bending_stiffness_data, stretching_stiffness_data, scale_factor_data, used_tags = visualize_stiffness.plot_all_data(kappa_path, stiffness_path, name, valid_tags, plot_data = False)

In [ ]:
max_bending_stiffness = np.max(bending_stiffness_data, axis = 1)
min_bending_stiffness = np.min(bending_stiffness_data, axis = 1)
max_stretching_stiffness = np.max(stretching_stiffness_data, axis = 1)
min_stretching_stiffness = np.min(stretching_stiffness_data, axis = 1)

In [ ]:
x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, name, valid_tags)

In [ ]:
min_scale_factors = np.min(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
max_scale_factors = np.max(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)

In [ ]:
angle_offsets = visualize_stiffness.get_max_flattening_factor_offset(stiffness_path, name, valid_tags)

In [ ]:
np.argmax(min_bending_stiffness)

In [ ]:
len(angles), len(radius)

In [ ]:
samples = np.array([[float(n) for n in tag.split('_')] for tag in valid_tags])

### Get scale function convex hull

In [ ]:
import matplotlib.cm as cm
import matplotlib as mpl

In [ ]:
from scipy.spatial import ConvexHull, convex_hull_plot_2d
import numpy as np
rng = np.random.default_rng()
points = rng.random((30, 2))   # 30 random points in 2-D
# points = np.concatenate((min_scale_factor.reshape((-1, 1)), max_scale_factor.reshape((-1, 1))), axis = 1)

points = np.concatenate((max_scale_factors.reshape((-1, 1)), min_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

In [ ]:
eqns = hull.equations

### Generate data without augmenting

In [ ]:
import visualize_stiffness, parametrization_helper

In [ ]:
stiffness_coefficients = np.array(visualize_stiffness.get_stiffness_coefficients(stiffness_path, name, (used_tags)))
# For patches with reflection symmetry:
stiffness_coefficients[:, 1] *= 0
stiffness_coefficients[:, 2] *= 0

In [ ]:
def get_stiffness_polynomial(s, theta):
    return s[0] * np.cos(theta)**2 * np.sin(theta)**2 + s[1] * np.cos(theta)**3 * np.sin(theta) + s[2] * np.cos(theta) * np.sin(theta)**3 + s[3] * np.cos(theta)**4 + s[4] * np.sin(theta)**4

In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [ ]:
grid_data = np.zeros((9, len(radius), len(angles)))

for i in range(len(valid_tags)):
    radius_index = int(i / len(angles))
    angles_index = int(i % len(angles))

    grid_data[0][radius_index][angles_index] = max_scale_factors[i]
    grid_data[1][radius_index][angles_index] = min_scale_factors[i]
    grid_data[2][radius_index][angles_index] = x_scale_factors[i]
    grid_data[3][radius_index][angles_index] = y_scale_factors[i]
    for s in range(5):
        grid_data[4 + s][radius_index][angles_index] = stiffness_coefficients[i][s]

In [ ]:
len(grid_data.shape) - 1

In [ ]:
grid_data.shape

In [ ]:

splines = parametrization_helper.scipy_get_mat_params_over_pattern_params_grid_interpolation(radius, angles, grid_data)

In [ ]:
scale_factors_grid_data = np.zeros((2, len(radius), len(angles)))
for i in range(len(valid_tags)):
    radius_index = int(i / len(angles))
    angles_index = int(i % len(angles))
    scale_factors_grid_data[0][radius_index][angles_index] = x_scale_factors[i]
    scale_factors_grid_data[1][radius_index][angles_index] = y_scale_factors[i]
scale_factors_splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(scale_factors_grid_data, radius, angles)

In [ ]:
radius, angles

In [ ]:
import numpy as np

# Generate 2D test parameters
x = np.linspace(0.5, 1.8, 100)
y = np.linspace(45, 75, 100)
test_parameters_x, test_parameters_y = np.meshgrid(x, y)

titles = ['max scale factors', 'min scale factors', 'x scale factors', 'y scale factors', 's1', 's2', 's3', 's4', 's5']
fig, axes = plt.subplots(1, 7, figsize=(45, 8))
index = [0, 1, 2, 3, 4, 7, 8]

for i in range(len(index)):
    
    # Evaluate the spline at the 2D test parameters
    z = splines[index[i] * 3 + 0]([test_parameters_x.flatten(), test_parameters_y.flatten()])
    z = z.reshape((100, 100))

    # Use imshow to visualize the 2D data
    im = axes[i].imshow(z, extent=[0.5, 2.3, 45, 75], origin='lower', aspect='auto', cmap='coolwarm')
    axes[i].set_title(titles[index[i]], fontsize=21)

    # Add a colorbar to each subplot
    fig.colorbar(im, ax=axes[i])

### End data generating

### Parametrization

In [ ]:
import sys; sys.path.append('../../../../'); sys.path.append('../../../../periodic_patches/'); sys.path.append('../../../experiments/'); sys.path.append('../../../../gmsh')
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [ ]:
sys.path.append('periodic_patches/')
sys.path.append('gmsh')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
from py_newton_optimizer import NewtonOptimizerOptions

In [ ]:
import MeshFEM, parallelism, benchmark, utils
parallelism.set_max_num_tbb_threads(32)
parallelism.set_gradient_assembly_num_threads(32)
parallelism.set_hessian_assembly_num_threads(32)

In [ ]:
import utils, mesh_utilities
importlib.reload(utils)

In [ ]:
target_surf = mesh.Mesh("../../../../SiggraphExamples/Meshes/20200106_teaser_srf_v3_remeshed.obj")
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=False))
# target_surf = mesh_utilities.subdivide_loop(target_surf, 1)
name = '20200106_teaser_srf_v3_remeshed.obj'

In [ ]:
lines = np.array(eqns)

### New local global with convex hull

In [ ]:
lg = parametrization.LocalGlobalGenericParametrizer(target_surf, parametrization.lscm(target_surf))

lg.setLines(eqns)
lg.alphaMin = hull.min_bound[0]
lg.alphaMax = hull.max_bound[0]

lg.betaMin = hull.min_bound[1]
lg.betaMax = hull.max_bound[1]

print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [ ]:
lg.alphaMin, lg.alphaMax, lg.betaMin, lg.betaMax

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(lg, show_main = True)

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, lg.getAlphas(), lg.getBetas())

### Pattern parameters optimization

In [ ]:
default_pattern_params = np.array([1.2]  * len(lg.getAlphas()) + [66]  * len(lg.getAlphas()))

In [ ]:
# mat_info = np.array(default_pattern_params).reshape((2, len(lg.getAlphas())))

In [ ]:
rparam = parametrization.RegularizedPatternParametrizer(lg, splines, default_pattern_params, len(grid_data.shape) - 1)
rparam.patternParamBounds = np.array([[0.5, 1.8], [48, 75]])
rparam.patternParamNormalizationFactors = np.array([1, 75 - 48])
rparam.diffRegW = 0.0

In [ ]:
rparam.patternRegP

In [ ]:
visualization.visualize_both(rparam, height = 4)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType

In [ ]:
rparam.bendRegW = 1

In [ ]:
rparam.energy(PET.RGP)

In [ ]:
rparam.energy(PET.Bending)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
import parametrization_optimization_helper

In [ ]:
num_pattern_params = 2

In [ ]:
parametrization_optimization_helper.initialize_pattern_parameters(rparam, lines, num_pattern_params)

In [ ]:
phiRegW = parametrization_optimization_helper.add_phi_regularization(rparam, lines, num_pattern_params)

In [ ]:
patternRegW = parametrization_optimization_helper.add_pattern_regularization(rparam, lines, num_pattern_params, phiRegW)

### Bending

In [ ]:
bendRegW = parametrization_optimization_helper.add_bending_energy(rparam, lines, num_pattern_params, phiRegW, patternRegW)

## Upsampling and channel generation

In [ ]:
import parametrization_helper
importlib.reload(parametrization_helper)

In [ ]:
mid_point = np.array([0, 5 / 4])
from shapely.geometry import LineString


def line_segment_intersection(segment1, segment2):
    line1 = LineString(segment1)
    line2 = LineString(segment2)
    intersection = line1.intersection(line2)

    if intersection.is_empty:
        return None
    else:
        return intersection.x, intersection.y
    
def fusing_curve_polyline(patternParams):
#     Draw dash_line
    r = patternParams[0]
    angle = patternParams[1]
    
    dash_point = np.array([np.cos(angle / 180 * np.pi), np.sin(angle / 180 * np.pi)]) * r + mid_point

    dash1 = np.array([dash_point, mid_point * 2 - dash_point])
    
    dash2 = np.array(dash1)
    dash2[:, 1] = - dash2[:, 1]

    # Check if the dashes intersect
    intersection = line_segment_intersection(dash1, dash2)
    if intersection is not None:
        # If they intersect, clip them at the intersection point
        dash1[1] = intersection
        dash2[1] = intersection
        
    return np.array([(dash1 + np.array([2.5, 2.5])) / 5 * np.pi, (dash2 + np.array([2.5, 2.5])) / 5 * np.pi])

In [ ]:
# fusing_curve_polyline([1.6225, 50.9199])

In [ ]:
fusing_lines = fusing_curve_polyline([1.2, 50.9199]).reshape(-1, 2)
fusing_edges = [[0, 1], [2, 3]]

In [ ]:
boundary_vertices = [[0, 0], [np.pi, 0], [np.pi, np.pi], [0, np.pi]]

In [ ]:
boundary_edges = np.array([[0, 1], [1, 2], [2, 3], [3, 0]]) + len(fusing_lines)

In [ ]:
visualization.plot_line_segments(list(fusing_lines) + boundary_vertices, fusing_edges + list(boundary_edges))

In [ ]:
sdfVertices, sdfTris, sdf, sheet_vxs, concatenated_polylines, sheet_edges_polylines,  boundaryVxs, boundaryEdges, upsampleMesh_vertices, upsampleMesh_triangles, upsampledAngles, upsampledPatternParams = parametrization_helper.get_polyline_from_pattern_parameters(rparam, fusing_curve_polyline, nsubdiv = 4, frequency=0.2, duplicates_removable_threshold=[1e-4, 1e-2, 1e-1, 1e0, 2e0])

In [ ]:
if len(boundaryEdges) > 1:
    concatenated_boundary_edges = []
    for polyline in boundaryEdges:
        concatenated_boundary_edges.extend(polyline)
    concatenated_boundary_edges = np.array(concatenated_boundary_edges)

    # Define a function to calculate the length of a sublist
def sublist_length(sublist):
    return len(sublist)

# Sort boundaryEdges in descending order of sublist length
boundaryEdges = sorted(boundaryEdges, key=sublist_length, reverse=True)


In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, width = 5, height=5)
visualization.plot_line_segments(sheet_vxs, concatenated_polylines, width = 15, height = 15)
visualization.plot_line_segments(list(sheet_vxs) + list(boundaryVxs), list(concatenated_polylines) + list(concatenated_boundary_edges + len(sheet_vxs)), width = 5, height = 5)
plt.scatter(boundaryVxs[concatenated_boundary_edges[:,0], 0], boundaryVxs[concatenated_boundary_edges[:,0], 1], c = np.arange(len(concatenated_boundary_edges[:, 0])), cmap = mpl.colormaps['Greys'])

## Meshing and inflation simulation

In [ ]:
import mesher_helper
importlib.reload(mesher_helper)

In [ ]:
import time
time_stamp = time.strftime("%Y_%m_%d_%H_%M")

In [ ]:
selected_elements = [np.array(sublist)[:, 0] for sublist in boundaryEdges[1:]]
holes_vxs = boundaryVxs[selected_elements]

In [ ]:
v, f, fusing_data = mesher_helper.generate_mesh_non_periodic(4, boundaryVxs[np.array(boundaryEdges[0])[:, 0]], holes_vxs, sheet_vxs, concatenated_polylines, gui = False)

In [ ]:
import numpy as np
import copy

# Use the function
new_v, new_f, new_fusing_without_boundary = parametrization_helper.remove_dangling_vertices(v, f - 1, fusing_data)
m = MeshFEM.mesh.Mesh(new_v, new_f)
new_fusing = copy.copy(new_fusing_without_boundary)
new_fusing[m.boundaryVertices()] = True

In [ ]:
# m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, SV, SE, triArea=1e0)


In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(new_fusing) == 1)[0], width=10, height=10)


In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, new_fusing)

### Save pattern

In [ ]:
import shapely
channelMargin = 1

In [ ]:
V = m.vertices()
polylines = isheet.fusedRegionBooleanIntersectSheetBoundary()
shapely_boundaryEdges = shapely.MultiLineString([V[p] for p in polylines])
#utils.save(boundaryEdges.buffer(channelMargin), 'test.pkl.gz')
bypasses = shapely_boundaryEdges.buffer(channelMargin)
if bypasses.geom_type == 'Polygon': bypasses = [bypasses] # we generally expect a multipolygon...
outerAirChannelPolygons = [shapely.ops.unary_union([shapely.Polygon(boundaryVxs[np.array(boundaryEdges[0])[:, 0]])] + list(bypasses.geoms))]

In [ ]:
smart_polygon = outerAirChannelPolygons[0]

In [ ]:
coords = np.array(smart_polygon.exterior.coords)

In [ ]:
len(coords)

In [ ]:
np.concatenate((coords[:-1], np.zeros((len(coords)-1, 1))), axis = 1)

In [ ]:
plt.scatter(coords[:, 0], coords[:, 1])

In [ ]:
polylines = []
for polyline in sheet_edges_polylines:
    polyline = np.array(polyline)
    polylines.append(sheet_vxs[np.array(list(polyline[:, 0]) + list([polyline[-1, 1]]))][:, :2].tolist())
parametrization_helper.save_to_obj(coords[:-1], polylines, 'teaser_{}_sheet_pattern_{}_margin_{}.obj'.format(name, time_stamp, channelMargin))

### End

In [ ]:
from mesh_utilities import SurfaceSampler, tubeRemesh


paramSampler = SurfaceSampler(np.pad(rparam.uv(), [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

isheet.getVars()

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [ ]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [ ]:
import boundaries
bdryVars = boundaries.getOuterBoundaryVars(isheet)
fixedVars = bdryVars

In [ ]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = [], 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [ ]:
isheet.pressure = 1e-5

In [ ]:
opts.niter = 10

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

### Then inflate

In [ ]:
isheet.pressure = 5e-4
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)


In [ ]:
isheet.pressure = 5e-2

opts.niter = 2000
opts.gradTol = 1e-7

import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()

In [ ]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(isheet)[:, 0], bins=1000);
plt.xlim(-0.04, 0.1);

In [ ]:
import gzip

In [ ]:
pickle.dump(isheet,  gzip.open("igloo_pattern_optimized_{}_low_frequency_with_bending_high_resolution.pkl.gz".format(time_stamp), 'wb'))

### Generate Fabrication Files

In [ ]:
old_to_new = np.arange(np.max(isheet.wallVertices()) + 1)

In [ ]:
old_to_new[isheet.wallVertices()] = np.arange(len(isheet.wallVertices()))

In [ ]:
from parametrization_helper import form_polylines

In [ ]:
result_vxs = isheet.restWallVertexPositions()
result_edges = old_to_new[isheet.wallBoundaryEdges()]
result_edges = form_polylines(result_edges.tolist())
concatenated_polylines = []
for polyline in result_edges:
    concatenated_polylines.extend(polyline)


In [ ]:
visualization.plot_line_segments(result_vxs, concatenated_polylines)